# Distributed training: DataParallel, DistributedDataParallel (DDP), and sharding

_PyTorch (a complete course)_

**Replicate the model on every GPU, split the batch, and all-reduce the gradients so all copies stay in sync — DDP is the standard way.**

Data parallelism in one sentence: put a full copy of the model on every GPU, give each copy a
       different slice of the batch, let each compute gradients on its slice, then average the gradients across all
       copies so every model stays identical.

       That averaging step is an all-reduce: a collective operation where every GPU contributes its gradient
       tensor, they are summed (then divided by N), and the same averaged result is handed back to every GPU. Because
       all copies start equal and apply the same averaged gradient, they take the same optimizer step and remain
       byte-for-byte identical forever. It is exactly the &ldquo;map then combine&rdquo; shape of a Spark job
       (spark-intro), specialized to gradients.

---

This notebook is a practice scaffold for the **Distributed training: DataParallel, DistributedDataParallel (DDP), and sharding** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
# ddp_train.py  —  launch with:
#   torchrun --standalone --nproc_per_node=4 ddp_train.py
# (one process per GPU; torchrun sets RANK / WORLD_SIZE / LOCAL_RANK)
import os
import torch
import torch.nn as nn
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.data.distributed import DistributedSampler
import torch.distributed as dist


def main():
    # --- 1. Join the process group -------------------------------------
    # 'nccl' is the fast GPU backend. torchrun provides the env vars.
    dist.init_process_group(backend="nccl")
    rank        = dist.get_rank()              # global process index (0..N-1)
    world_size  = dist.get_world_size()        # total number of GPUs (N)
    local_rank  = int(os.environ["LOCAL_RANK"])# GPU index on this machine
    torch.cuda.set_device(local_rank)          # pin this process to its GPU
    device = torch.device("cuda", local_rank)

    torch.manual_seed(0)                       # same seed -> identical init

    # --- 2. A tiny synthetic dataset -----------------------------------
    X = torch.randn(8000, 784)
    y = torch.randint(0, 10, (8000,))
    dataset = TensorDataset(X, y)

    # DistributedSampler hands each rank a DISJOINT shard of the data.
    # Without it, every GPU would see the SAME batches (a classic bug).
    sampler = DistributedSampler(dataset, num_replicas=world_size,
                                 rank=rank, shuffle=True)
    local_batch = 32
    loader = DataLoader(dataset, batch_size=local_batch, sampler=sampler,
                        drop_last=True)        # drop_last keeps step counts
                                               # equal -> no all-reduce deadlock
    eff_batch = local_batch * world_size       # effective batch size
    if rank == 0:
        print(f"world_size={world_size}  local_batch={local_batch}  "
              f"effective_batch={eff_batch}")

    # --- 3. Model, wrapped in DDP --------------------------------------
    model = nn.Sequential(
        nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 10)
    ).to(device)
    model = DDP(model, device_ids=[local_rank])   # broadcasts weights from
                                                  # rank 0; averages grads via
                                                  # all-reduce in backward()

    # Linear scaling rule: scale base lr by the number of GPUs.
    base_lr = 0.05
    optimizer = torch.optim.SGD(model.parameters(),
                                lr=base_lr * world_size, momentum=0.9)
    loss_fn = nn.CrossEntropyLoss()            # expects raw logits + class idx

    # --- 4. The per-rank training loop ---------------------------------
    for epoch in range(5):
        sampler.set_epoch(epoch)               # reshuffle differently each epoch
        model.train()
        running = 0.0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)                 # forward on this rank's shard
            loss = loss_fn(logits, yb)
            loss.backward()                    # DDP all-reduces grads HERE
            optimizer.step()                   # every rank takes the same step
            running += loss.item()
        # Only rank 0 logs (otherwise N copies of every line).
        if rank == 0:
            print(f"epoch {epoch}  loss {running / len(loader):.4f}")

    # --- 5. Save ONCE, from rank 0 only --------------------------------
    if rank == 0:
        # .module unwraps the DDP wrapper to the plain model's state_dict
        torch.save(model.module.state_dict(), "model.pt")
        print("saved checkpoint from rank 0")

    dist.destroy_process_group()               # clean shutdown


if __name__ == "__main__":
    main()

## Visualize the data & results

_How much faster does DDP train as you add GPUs? Throughput (images/sec) and speedup vs the number of GPUs under a simple Amdahl-style model with communication overhead — near-linear at first, then sub-linear as all-reduce cost grows._

In [ ]:
import numpy as np

# Amdahl-style per-epoch time model with all-reduce communication overhead.
#   t(N) = t1 * (s + (1-s)/N)  +  c*(N-1)
# s = serial (non-parallelizable) fraction; c = per-step comm cost.
t1 = 1.0          # single-GPU epoch time (normalized)
s  = 0.02         # 2% of work is serial (data loading, fixed overhead)
c  = 0.0008 * t1  # communication grows with the number of GPUs

Ns = np.array([1, 2, 4, 8, 16, 32])
t  = t1 * (s + (1 - s) / Ns) + c * (Ns - 1)

base_throughput = 1000.0                 # images/sec on 1 GPU
throughput = base_throughput * (t1 / t)  # faster epoch -> higher throughput
ideal      = base_throughput * Ns        # perfect linear scaling
speedup    = t1 / t

for n, thr, sp in zip(Ns, throughput, speedup):
    print(f"N={n:>2d}  throughput={thr:8.1f} img/s  speedup={sp:5.2f}x "
          f"(ideal {n}x)")
# N= 1  throughput=  1000.0 img/s  speedup= 1.00x (ideal 1x)
# N= 2  throughput=  1958.0 img/s  speedup= 1.96x (ideal 2x)
# N= 4  throughput=  3740.6 img/s  speedup= 3.74x (ideal 4x)
# N= 8  throughput=  6752.4 img/s  speedup= 6.75x (ideal 8x)
# N=16  throughput= 10724.2 img/s  speedup=10.72x (ideal 16x)
# N=32  throughput= 13257.6 img/s  speedup=13.26x (ideal 32x)

import matplotlib.pyplot as plt
plt.plot(Ns, ideal, "--", color="#8b949e", label="ideal (linear)")
plt.plot(Ns, throughput, "o-", color="#4ea1ff", label="DDP (modeled)")
plt.xlabel("number of GPUs (N)")
plt.ylabel("throughput (images/sec)")
plt.title("DDP throughput vs number of GPUs")
plt.legend()
plt.show()

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A teammate scales from 1 GPU to 4 GPUs with DDP, but final accuracy drops noticeably versus the single-GPU baseline. They kept everything else the same. What is the most likely cause, and what is the fix?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note that 4-GPU DDP makes the effective batch size 4&times; larger (each GPU keeps its local batch, and gradients are averaged across all 4). — _DDP is mathematically one run on the concatenated big batch, so the optimizer is now stepping on a much larger batch than the baseline used._
- Realize a bigger batch gives a less noisy gradient, so the old learning rate is now effectively too small for the new batch. — _The learning rate was tuned for the small-batch gradient noise; at 4&times; the batch it no longer matches._
- Apply the linear scaling rule: multiply the learning rate by 4 (with a short warmup). — _Scaling lr with the effective batch restores the step magnitude the optimizer expects, recovering the baseline accuracy._

**Answer:** They forgot to scale the learning rate. With 4 GPUs the effective batch is 4&times; larger, which lowers gradient noise and makes the original learning rate effectively too small. Use the linear scaling rule &mdash; multiply the learning rate by 4 (the number of GPUs), typically with a few warmup epochs so the larger early steps stay stable. This is the single most common reason a correct DDP setup underperforms its single-GPU baseline.

</details>

**Problem 2.** A DDP job launched with torchrun --nproc_per_node=4 hangs partway through an epoch with no error &mdash; the GPUs sit at 100% then go idle and nothing happens. What class of bug is this, and what causes it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recognize that DDP does an all-reduce on every backward pass, which is a synchronization barrier: all 4 ranks must reach it together. — _If even one rank does not arrive at an all-reduce, the others block waiting for it indefinitely &mdash; a deadlock, which looks like a silent hang._
- Look for a reason one rank runs fewer steps than the others &mdash; an uneven dataset split, drop_last=False leaving a ragged last batch, or a per-rank early break. — _Different step counts mean one rank finishes its loop while the others are still waiting at the next all-reduce._
- Make the step counts equal: set drop_last=True in the DataLoader (or use DDP's join() context for uneven inputs). — _Equal step counts keep every rank hitting each all-reduce together, so no rank is left waiting._

**Answer:** It is a deadlock from uneven batch counts. DDP synchronizes via all-reduce on every backward pass, so all ranks must run the same number of steps. When one rank runs out of batches early &mdash; ragged last batch, uneven split, or a per-rank break &mdash; the others block forever at the next all-reduce and the job hangs with no error. Fix it by forcing equal step counts: drop_last=True in the DataLoader, or wrap the loop in DDP's model.join() context to handle uneven inputs.

</details>

**Problem 3.** You have one big model and a large dataset. You try DDP and immediately get a CUDA out-of-memory error before training even starts &mdash; the model will not even fit one copy on one GPU. Is DDP the right tool, and what should you use instead?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Identify which wall you hit: this is the memory wall (the model does not fit), not the compute wall (training too slow). — _DDP is data parallelism &mdash; it puts a full replica on every GPU. If one replica does not fit, DDP cannot help._
- Recognize you need to shard the model itself &mdash; split its weights, gradients, and optimizer state across GPUs rather than replicating them. — _Sharding means no single GPU ever holds the whole model, so a model larger than one GPU's memory becomes trainable._
- Switch to Fully Sharded Data Parallel (FSDP) or DeepSpeed/ZeRO. — _These shard parameters/optimizer state across GPUs (and can offload to CPU), fitting models too big for plain DDP while still scaling across data._

**Answer:** DDP is the wrong tool here. DDP replicates a full copy of the model on every GPU, so if one copy will not fit, it cannot run &mdash; this is the memory wall, not the speed problem DDP solves. You need model sharding: Fully Sharded Data Parallel (FSDP) or DeepSpeed/ZeRO, which split the weights, gradients, and optimizer state across GPUs (optionally offloading to Central Processing Unit memory) so no single GPU ever holds the whole model. Reserve plain DDP for when the model fits but training is too slow.

</details>